# Calibration check — EuroBERT

**Question:** when the model says 80% confident, is it right 80% of the time?

Approach:
1. Load val predictions (`artifacts/predictions/{run_name}_val.parquet`)
2. Pool every (deed, code) score as one yes/no decision
3. Plot a reliability diagram (predicted confidence vs actual accuracy)
4. If miscalibrated, fit temperature scaling and re-plot
5. Write calibrated scores back into the prediction contract

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from scipy.optimize import minimize_scalar

from shared.config import RESULTS, ARTIFACTS
from shared.io      import read_predictions

RUN_NAME = "focal"   # change this to calibrate a different run
VAL_METHOD = f"{RUN_NAME}_val"

print(f"Calibrating: {RUN_NAME}")

## 1. Load val predictions and the true labels

In [ ]:
from shared.data   import load_train_test
from shared.config import TRAIN_PATH, TEST_PATH
from shared.splits import load_split
from shared.labels  import get_mlb

train_full_df, test_df = load_train_test(TRAIN_PATH, TEST_PATH)
mlb, classes, num_classes = get_mlb()
train_idx, val_idx = load_split()
val_df = train_full_df.iloc[val_idx].reset_index(drop=True)
y_val  = mlb.transform(val_df["rechtsfeitcodes"]).astype(np.float32)

# load val predictions (long format) and pivot to wide matrix aligned with y_val
pred_df = read_predictions(VAL_METHOD)

code_to_idx = {c: i for i, c in enumerate(classes)}
akteid_to_idx = {a: i for i, a in enumerate(val_df["akteId"].tolist())}

scores = np.zeros((len(val_df), num_classes), dtype=np.float32)
for row in pred_df.itertuples(index=False):
    i = akteid_to_idx.get(row.akteId)
    j = code_to_idx.get(row.code)
    if i is not None and j is not None:
        scores[i, j] = row.score

print(f"scores shape: {scores.shape}")
print(f"y_val shape : {y_val.shape}")

## 2. Pool every (deed, code) pair as one binary decision
Flatten the matrices — every cell becomes one yes/no data point.

In [ ]:
y_true_flat  = y_val.flatten()
scores_flat  = scores.flatten()

print(f"Total (deed, code) pairs: {len(y_true_flat):,}")
print(f"Positive rate (all pairs): {y_true_flat.mean():.4f}")

# Most (deed, code) pairs are trivial true negatives (model correctly says ~0%).
# These dominate the pool and make the reliability diagram uninformative —
# everything piles into the bottom-left corner. Filter to pairs where the
# model expressed some non-trivial belief, so calibration reflects the
# decisions that actually matter for the auto-accept / escalate framework.
SCORE_FLOOR = 0.05
mask = scores_flat >= SCORE_FLOOR

y_true_flat  = y_true_flat[mask]
scores_flat  = scores_flat[mask]

print(f"\nAfter filtering to scores >= {SCORE_FLOOR}:")
print(f"  Pairs remaining : {len(y_true_flat):,}")
print(f"  Positive rate   : {y_true_flat.mean():.4f}")

## 3. Reliability diagram — before calibration

In [ ]:
frac_pos, mean_pred = calibration_curve(y_true_flat, scores_flat, n_bins=10, strategy="uniform")

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
ax.plot(mean_pred, frac_pos, marker="o", label="Before calibration")
ax.set_xlabel("Predicted confidence")
ax.set_ylabel("Fraction actually correct")
ax.set_title(f"Reliability diagram — {RUN_NAME} (val set)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

out_path = RESULTS / "plots" / f"{RUN_NAME}_reliability_before.png"
out_path.parent.mkdir(exist_ok=True)
fig.savefig(out_path, dpi=150)
print(f"Saved → {out_path}")
plt.show()

## 4. Fit temperature scaling

Temperature scaling divides the **logit** (not the probability) by a single
scalar T, then re-applies sigmoid. T > 1 softens overconfident predictions
towards 0.5; T < 1 sharpens them. T is tuned on val to minimise mean absolute calibration error (MACE)

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def logit(p, eps=1e-7):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def mace_for_temperature(T, logits, y_true, n_bins=10):
    """Mean absolute calibration error: average |confidence - accuracy| across bins."""
    scaled_probs = sigmoid(logits / T)
    frac_pos, mean_pred = calibration_curve(y_true, scaled_probs, n_bins=n_bins, strategy="uniform")
    return np.mean(np.abs(mean_pred - frac_pos))

logits_flat = logit(scores_flat)

# grid search over T, pick the one with lowest MACE
T_candidates = np.arange(0.3, 5.01, 0.1)
mace_scores  = [mace_for_temperature(T, logits_flat, y_true_flat) for T in T_candidates]

T_optimal = T_candidates[np.argmin(mace_scores)]
best_mace = min(mace_scores)

print(f"MACE-optimal temperature T = {T_optimal:.2f}  (MACE = {best_mace:.4f})")
print(f"(T > 1 means the model was overconfident; T < 1 means underconfident)")

## 5. Reliability diagram — after calibration

In [ ]:
scores_calibrated_flat = sigmoid(logits_flat / T_optimal)

frac_pos_cal, mean_pred_cal = calibration_curve(y_true_flat, scores_calibrated_flat, n_bins=10, strategy="uniform")

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
ax.plot(mean_pred, frac_pos, marker="o", alpha=0.5, label="Before calibration")
ax.plot(mean_pred_cal, frac_pos_cal, marker="o", label=f"After calibration (T={T_optimal:.2f})")
ax.set_xlabel("Predicted confidence")
ax.set_ylabel("Fraction actually correct")
ax.set_title(f"Reliability diagram — {RUN_NAME} (val set)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

out_path = RESULTS / "plots" / f"{RUN_NAME}_reliability_after.png"
fig.savefig(out_path, dpi=150)
print(f"Saved → {out_path}")
plt.show()

## 6. Write calibrated scores into the prediction contract

In [ ]:
import json as _json

# Load per-class thresholds saved by bert.ipynb during training
thresh_path = ARTIFACTS / "models" / RUN_NAME / "thresholds.json"
if thresh_path.exists():
    with open(thresh_path) as _f:
        thresh_dict = _json.load(_f)
    thresholds = np.array([thresh_dict.get(str(c), 0.5) for c in classes], dtype=np.float32)
    print(f"Loaded per-class thresholds from {thresh_path}")
    print(f"  min={thresholds.min():.3f}  max={thresholds.max():.3f}  mean={thresholds.mean():.3f}")
else:
    thresholds = np.full(num_classes, 0.5, dtype=np.float32)
    print(f"WARNING: thresholds.json not found at {thresh_path} — falling back to 0.5")

# --- calibrate VAL set predictions ---
val_logits            = logit(scores)   # scores = val matrix loaded in cell 3
val_scores_calibrated = sigmoid(val_logits / T_optimal)
val_predicted         = val_scores_calibrated > thresholds[np.newaxis, :]

write_predictions(
    akteIds   = val_df["akteId"].tolist(),
    scores    = val_scores_calibrated,
    predicted = val_predicted,
    classes   = classes,
    method    = VAL_METHOD,
)
print(f"Calibrated VAL scores written to artifacts/predictions/{VAL_METHOD}.parquet")

# --- calibrate TEST set predictions ---
test_pred_df = read_predictions(RUN_NAME)

akteid_to_idx_test = {a: i for i, a in enumerate(test_df["akteId"].tolist())}
test_scores = np.zeros((len(test_df), num_classes), dtype=np.float32)
for row in test_pred_df.itertuples(index=False):
    i = akteid_to_idx_test.get(row.akteId)
    j = code_to_idx.get(row.code)
    if i is not None and j is not None:
        test_scores[i, j] = row.score

test_logits            = logit(test_scores)
test_scores_calibrated = sigmoid(test_logits / T_optimal)
test_predicted         = test_scores_calibrated > thresholds[np.newaxis, :]

write_predictions(
    akteIds   = test_df["akteId"].tolist(),
    scores    = test_scores_calibrated,
    predicted = test_predicted,
    classes   = classes,
    method    = RUN_NAME,
)
print(f"Calibrated TEST scores written to artifacts/predictions/{RUN_NAME}.parquet")


## 7. Append calibration section to SUMMARY.md

In [ ]:
summary = f"""
## Calibration — {RUN_NAME}

**Method:** temperature scaling on validation set (pooled across all deed×code pairs).

**Optimal temperature:** T = {T_optimal:.4f}

{'Model was overconfident (T > 1) — scores pulled towards 0.5.' if T_optimal > 1 else 'Model was underconfident (T < 1) — scores pushed away from 0.5.' if T_optimal < 1 else 'Model was already well-calibrated.'}

See `results/plots/{RUN_NAME}_reliability_before.png` and `_after.png` for the reliability diagrams.

Calibrated scores have been written to `artifacts/predictions/{RUN_NAME}.parquet`.

---
"""

summary_path = RESULTS / "SUMMARY.md"
with summary_path.open("a", encoding="utf-8") as f:
    f.write(summary)

print(f"SUMMARY.md updated → {summary_path}")